In [9]:
import pandas as pd

roll_no = "1024170271"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# last two digits of my roll number are 7 and 1
# categories = ["billing", "account", "general"]
# 7 % 3 = 1 -> account
# 1 % 3 = 1 -> account
# so both my personalized entries fall under the "account" category

personalized_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Settings > Profile > Update Mobile Number.",
     "keywords": "mobile number update", "category": "account"},
    {"question": "how do i change my registered email address",
     "answer": "Go to Settings > Profile > Update Email.",
     "keywords": "email change update", "category": "account"},
]

all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)
print(df)

                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5  how do i change my registered email address   

                                             answer                keywords  \
0                         The annual fee is Rs 500.   fee cost price charge   
1                  Go to Settings > Reset Password.    password reset login   
2                         We are open 9 AM to 5 PM.  hours timing open time   
3        You can pay via UPI, card, or net banking.     pay payment upi fee   
4  Go to Settings > Profile > Update Mobile Number.    mobile number update   
5          Go to Settings > Profile > Update Email.     email change update   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  account  
5  account  


In [10]:
# this function checks how many keywords of each FAQ entry match the words in the user's query
# more matching keywords = higher score = more confident this is the right answer

def score_query(query, df):
    query_words = query.lower().split()
    results = []
    for i in range(len(df)):
        row_keywords = df.loc[i, "keywords"].split()
        score = 0
        for word in query_words:
            if word in row_keywords:
                score += 1
        if score > 0:
            results.append((df.loc[i, "question"], df.loc[i, "answer"], score))
    results.sort(key=lambda x: x[2], reverse=True)
    return results

matches = score_query("what is the annual fee", df)
for question, answer, score in matches:
    print(question, "|", answer, "| confidence score:", score)

what is the annual fee | The annual fee is Rs 500. | confidence score: 1
how can i pay the fee | You can pay via UPI, card, or net banking. | confidence score: 1


## Q3 - Filtering by category

In [11]:
def same_category(category_name, df):
    return df[df["category"] == category_name]

# using "account" since it's the category of my personalized entries
account_questions = same_category("account", df)
print(account_questions)

                                      question  \
1                        how to reset password   
4  how do i update my registered mobile number   
5  how do i change my registered email address   

                                             answer              keywords  \
1                  Go to Settings > Reset Password.  password reset login   
4  Go to Settings > Profile > Update Mobile Number.  mobile number update   
5          Go to Settings > Profile > Update Email.   email change update   

  category  
1  account  
4  account  
5  account  


## Q4 - Adding a keyword and saving to CSV

In [12]:
# picking entry 0 (the annual fee question) to add a new keyword to
new_keyword = input("Enter a new keyword to add: ")
df.loc[0, "keywords"] = df.loc[0, "keywords"] + " " + new_keyword
print(df.loc[0, "keywords"])

df.to_csv(roll_no + "_faq_data.csv", index=False)
print("Saved as", roll_no + "_faq_data.csv")

Enter a new keyword to add:  annual


fee cost price charge annual
Saved as 1024170271_faq_data.csv


## Q5 - Count of entries per category

In [13]:
print(df.groupby("category")["question"].count())

category
account    3
billing    2
general    1
Name: question, dtype: int64


## Q6 - Handling ties in the scoring function

In [14]:
def score_query_with_ties(query, df):
    query_words = query.lower().split()
    results = []
    for i in range(len(df)):
        row_keywords = df.loc[i, "keywords"].split()
        score = 0
        for word in query_words:
            if word in row_keywords:
                score += 1
        if score > 0:
            results.append((df.loc[i, "question"], df.loc[i, "answer"], score))

    if len(results) == 0:
        print("No matching entries found.")
        return

    max_score = max(r[2] for r in results)
    top_matches = [r for r in results if r[2] == max_score]

    if len(top_matches) > 1:
        print("Tie detected! Showing all top matches instead of picking one:")
        for question, answer, score in top_matches:
            print(question, "|", answer, "| score:", score)
    else:
        question, answer, score = top_matches[0]
        print("Best match:", question, "|", answer, "| score:", score)

# this query matches both fee-related entries equally, so it should show a tie
print("Query 1: cost pay")
score_query_with_ties("cost pay", df)

print()

# this query only matches one entry clearly, so no tie here
print("Query 2: how to reset password")
score_query_with_ties("how to reset password", df)

Query 1: cost pay
Tie detected! Showing all top matches instead of picking one:
what is the annual fee | The annual fee is Rs 500. | score: 1
how can i pay the fee | You can pay via UPI, card, or net banking. | score: 1

Query 2: how to reset password
Best match: how to reset password | Go to Settings > Reset Password. | score: 2
